In [ ]:
import pandas as pd
import numpy as np
import re

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [ ]:
from tensorflow.keras.models import load_model

model = load_model("/content/drive/MyDrive/Colab Notebooks/UIT/model_2.keras")

In [ ]:
df = pd.read_csv("/content/spa.txt", sep="\t", header=None)
df.columns = ["English", "Spanish", "Meta"]

df = df[["English", "Spanish"]]
df.head(5)

,English,Spanish
0,Go.,Ve.
1,Go.,Vete.
2,Go.,Vaya.
3,Go.,Váyase.
4,Go.,Id.


In [ ]:
def clean(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Zñáéíóúü ]", "", text)
    return text

df["English"] = df["English"].apply(clean)
df["Spanish"] = df["Spanish"].apply(clean)

df.head()

,English,Spanish
0,go,ve
1,go,vete
2,go,vaya
3,go,váyase
4,go,id


In [ ]:
eng_tokenizer = Tokenizer()
sp_tokenizer = Tokenizer()

eng_tokenizer.fit_on_texts(df["English"])
sp_tokenizer.fit_on_texts(df["Spanish"])

eng_seq = eng_tokenizer.texts_to_sequences(df["English"])
sp_seq = sp_tokenizer.texts_to_sequences(df["Spanish"])

In [ ]:
max_len = 15

eng_seq = pad_sequences(eng_seq, maxlen=max_len, padding="post")
sp_seq = pad_sequences(sp_seq, maxlen=max_len, padding="post")

In [ ]:
from tensorflow.keras.layers import Input

model = Sequential()

model.add(Input(shape=(max_len,)))

model.add(Embedding(len(eng_tokenizer.word_index)+1, 128))
model.add(LSTM(128, return_sequences=True))
model.add(Dense(len(sp_tokenizer.word_index)+1, activation="softmax"))

model.summary()


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ (None, 15, 128)        │     1,835,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, 15, 128)        │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 15, 28253)      │     3,644,637 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,611,741 (21.41 MB)

 Trainable params: 5,611,741 (21.41 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
sp_seq = np.expand_dims(sp_seq, -1)

In [ ]:
import pickle
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy"
)
model.fit(eng_seq, sp_seq, epochs=5, batch_size=64)

model.save("/content/drive/MyDrive/Colab Notebooks/UIT/model_2.keras")

with open("/content/drive/MyDrive/Colab Notebooks/UIT/eng_tokenizer.pkl", "wb") as f:
    pickle.dump(eng_tokenizer, f)

with open("/content/drive/MyDrive/Colab Notebooks/UIT/sp_tokenizer.pkl", "wb") as f:
    pickle.dump(sp_tokenizer, f)

print("Model and tokenizers saved successfully!")

Epoch 1/5
2254/2254 ━━━━━━━━━━━━━━━━━━━━ 124s 54ms/step - loss: 1.6813
Epoch 2/5
2254/2254 ━━━━━━━━━━━━━━━━━━━━ 125s 55ms/step - loss: 1.5763
Epoch 3/5
2254/2254 ━━━━━━━━━━━━━━━━━━━━ 125s 55ms/step - loss: 1.5030
Epoch 4/5
2254/2254 ━━━━━━━━━━━━━━━━━━━━ 124s 55ms/step - loss: 1.4448
Epoch 5/5
2254/2254 ━━━━━━━━━━━━━━━━━━━━ 124s 55ms/step - loss: 1.3968
Model and tokenizers saved successfully!


In [ ]:
index_to_word = {i: w for w, i in sp_tokenizer.word_index.items()}

def translate(sentence):
    sentence = clean(sentence)

    seq = eng_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=max_len, padding="post")

    pred = model.predict(seq)[0]

    words = []
    for p in pred:
        word = index_to_word.get(np.argmax(p), "")
        words.append(word)

    return " ".join(words).strip()

In [ ]:
while True:
    user_input = input("\nEnter English sentence (or 'quit'): ")

    if user_input.lower() == "quit":
        break

    translation = translate(user_input)

    print("Spanish Translation:", translation)


Enter English sentence (or 'quit'): hi
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step
Spanish Translation: hola

Enter English sentence (or 'quit'): thanks
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Spanish Translation: gracias

Enter English sentence (or 'quit'): welcome
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Spanish Translation: bienvenido

Enter English sentence (or 'quit'): i'm done
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Spanish Translation: estoy hecho

Enter English sentence (or 'quit'): i'm rich
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Spanish Translation: estoy rico

Enter English sentence (or 'quit'): let's go
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Spanish Translation: vamos a
